###  Installation and imports

In [ ]:
!pip install backtesting --quiet

from pathlib import Path
import os, glob, re, warnings, itertools
import joblib
import numpy as np
import pandas as pd
from backtesting import Backtest, Strategy

warnings.filterwarnings('ignore')

###  Paths and bundle loading

In [ ]:
# ============================================================
# ASSET SELECTION (TICKERS)
# ============================================================
TICKERS_TO_TEST = ['ETHUSDT', 'BNBUSDT', 'ADAUSDT']

# ============================================================
# STRATEGY PARAMETER GRID — Restrained to 3 Configurations
# ============================================================
STRATEGY_GRID = [
    # 1. Baseline configuration
    {'entry_mode': 'immediate', 'stop_pct': 0.03, 'min_r_multiple': 3.0,
     'move_to_breakeven_r': 1.0, 'trail_activation_r': 2.0, 'trail_distance_r': 0.5},
    # 2. Expanded stop-loss threshold — designed to mitigate market noise premature liquidation
    {'entry_mode': 'immediate', 'stop_pct': 0.05, 'min_r_multiple': 3.0,
     'move_to_breakeven_r': 1.0, 'trail_activation_r': 2.0, 'trail_distance_r': 0.5},
    # 3. Breakout entry execution combined with an expanded stop-loss threshold
    {'entry_mode': 'breakout',  'stop_pct': 0.05, 'min_r_multiple': 4.0,
     'move_to_breakeven_r': 1.0, 'trail_activation_r': 2.0, 'trail_distance_r': 0.5},
]

# ============================================================
# PROBABILITY MATRIX — 4-Hour Forecast Horizon
# ============================================================
PROBA_COL  = 'proba_4h'
PROBA_GRID = [0.24, 0.45, 0.65]     # 0.24 — Optimally tuned threshold for the 4h horizon derived from the asset bundle

# ============================================================
# MISCELLANEOUS SIMULATION PARAMETERS
# ============================================================
CASH       = 10_000
COMMISSION = 0.0005
TRADE_DIR  = 'both'
USE_PPZ    = True


### Paths and bundle loading

In [ ]:
BUNDLE_PATH = '/kaggle/input/models/roboeggs/lightgbm/scikit-learn/default/1/best_model_bundle.joblib'
KLINES_BASE = '/kaggle/input/binance-spot-klines-usdt/data/spot/monthly/klines'

if not Path(BUNDLE_PATH).exists():
    hits = glob.glob('/kaggle/input/**/best_model_bundle.joblib', recursive=True)
    if not hits:
        raise FileNotFoundError('best_model_bundle.joblib not found')
    BUNDLE_PATH = hits[0]

DATA_PATH = glob.glob('/kaggle/input/**/breakout_features.parquet', recursive=True)[0]

bundle = joblib.load(BUNDLE_PATH)
models        = bundle['models']
feature_cols  = bundle['feature_cols']
THRESHOLDS    = bundle['thresholds']
FOCUS_TICKERS = bundle['focus_tickers']

print('Bundle :', BUNDLE_PATH)
print('Focus  :', FOCUS_TICKERS)
print('Thresh :', THRESHOLDS)

testable = sorted(set(TICKERS_TO_TEST) & set(FOCUS_TICKERS))
skipped  = sorted(set(TICKERS_TO_TEST) - set(FOCUS_TICKERS))
print('\nWill test :', testable)
print('Skipped   :', skipped or '—')

Bundle : /kaggle/input/models/roboeggs/lightgbm/scikitlearn/default/1/best_model_bundle.joblib
Focus  : ['BTTCUSDT', 'PAXGUSDT', 'BNBUSDT', 'LTCUSDT', 'XRPUSDT', 'XLMUSDT', 'ETHUSDT', 'ADAUSDT', 'DOGEUSDT', 'HBARUSDT']
Thresh : {'breakout_1h': 0.13999999999999999, 'breakout_4h': 0.24, 'breakout_24h': 0.33999999999999997}

Will test : ['ADAUSDT', 'BNBUSDT', 'ETHUSDT']
Skipped   : —


### Data loading and cleaning

In [65]:
TARGETS = ['breakout_1h', 'breakout_4h', 'breakout_24h']

df = pd.read_parquet(DATA_PATH)
df['prediction_time'] = pd.to_datetime(df['prediction_time'], utc=True, errors='coerce')
df = df.dropna(subset=['ticker', 'level_id', 'prediction_time']).copy()
df['ticker'] = df['ticker'].astype(str).str.upper()

stable_symbols = {
    'USDC','USDT','TUSD','BUSD','FDUSD','DAI','USDP','PAX','EUR',
    'GBP','USDE','USDS','UST','USTC','FRAX','PYUSD','EURC',
}
leveraged_re    = re.compile(r'(UP|DOWN|BULL|BEAR)(USDT|USDC|BUSD)?$')
quote_suffix_re = re.compile(r'(USDT|USDC|BUSD)$')

def is_excluded_ticker(t):
    base = quote_suffix_re.sub('', t)
    return t in stable_symbols or base in stable_symbols or bool(leveraged_re.search(t))

df = df.loc[~df['ticker'].map(is_excluded_ticker)].copy()
counts = df['ticker'].value_counts()
df = df.loc[df['ticker'].isin(counts[counts >= 50].index)].copy()

df['level_key'] = df['ticker'].astype(str) + '::' + df['level_id'].astype(str)
df = df.sort_values(['prediction_time', 'ticker', 'level_id']).reset_index(drop=True)
for col in ['touch_times', 'time_since_last_touch']:
    if col in df.columns:
        df = df.drop(columns=col)

if 'price_position_in_zone' in df.columns:
    raw = pd.to_numeric(df['price_position_in_zone'], errors='coerce')
    df['is_degenerate_zone'] = raw.isna().astype('int8')
    df['price_position_in_zone'] = raw.clip(0.0, 1.0).fillna(0.5)
else:
    df['is_degenerate_zone'] = 0

LEVEL_TYPE_MAP = {'resistance': 0, 'support': 1, 'missing': -1}
df['level_type'] = (df['level_type'].fillna('missing').astype(str)
                    .map(LEVEL_TYPE_MAP).fillna(-1).astype('int16'))

model_df = df.loc[df['ticker'].isin(FOCUS_TICKERS)].copy()
print(f'Rows in focus subset: {len(model_df):,}')

Rows in focus subset: 36,866


### The same chronological split as in train.

In [66]:
time_values = model_df['prediction_time'].sort_values()
train_cutoff = time_values.quantile(0.70)
val_cutoff   = time_values.quantile(0.85)

def time_bucket(ts):
    if ts <= train_cutoff:  return 'train'
    if ts <= val_cutoff:    return 'val'
    return 'test'

model_df['_time_bucket'] = model_df['prediction_time'].map(time_bucket)
gbc = model_df.groupby('level_key')['_time_bucket'].nunique()
clean_groups = gbc[gbc == 1].index
model_df = model_df.loc[model_df['level_key'].isin(clean_groups)].copy()

test = model_df.loc[
    (model_df['_time_bucket'] == 'test') &
    (model_df['ticker'].isin(testable))
].copy()

print(f'Test rows: {len(test):,}')
print(f'Range    : {test.prediction_time.min()} → {test.prediction_time.max()}')
print(f'Tickers  : {sorted(test.ticker.unique())}')

Test rows: 1,056
Range    : 2024-12-09 14:48:00+00:00 → 2025-12-30 20:59:00+00:00
Tickers  : ['ADAUSDT', 'BNBUSDT', 'ETHUSDT']


### Feature compliance check
The classes _VirtualLevel, _SignalDetector, and SignalStrategy do not change — they work with any CSV that contains prediction_time, direction, proba_1h/4h/24h, center, zone_low/high, and price_position_in_zone. Copy them from the old notebook, one-to-one.

In [67]:
# Verify the feature contract with the bundle
missing = [c for c in feature_cols if c not in test.columns]
assert not missing, f'Missing feature columns: {missing}'

# Check the inner LightGBM sees the same order
def get_inner(m):
    try:    return m.estimator.estimator_
    except AttributeError: pass
    try:    return m.estimator_
    except AttributeError: pass
    return None

m0 = get_inner(models['breakout_1h'])
if m0 is not None:
    try:
        names = list(m0.feature_name_)
        assert names == feature_cols, 'Feature order mismatch'
        print('Feature order verified against bundle:', len(names))
    except AttributeError:
        print('feature_name_ not exposed, relying on saved order')

print('Bundle features  :', len(feature_cols))
print('Test cols present:', sum(c in test.columns for c in feature_cols))

Bundle features  : 45
Test cols present: 45


### Signal generation with calibrated probabilities

In [68]:
X_test = test[feature_cols]
proba = {t: models[t].predict_proba(X_test)[:, 1] for t in TARGETS}

base = test[[
    'prediction_time','ticker','level_id','level_type',
    'center','zone_low','zone_high','price_position_in_zone',
    'touch_count','is_confirmation_touch',
]].copy()

base['proba_1h']  = proba['breakout_1h']
base['proba_4h']  = proba['breakout_4h']
base['proba_24h'] = proba['breakout_24h']

base['level_type_str'] = base['level_type'].map({0: 'resistance', 1: 'support'})
base['direction'] = base['level_type_str'].map({'resistance': 'long', 'support': 'short'})
base = base.dropna(subset=['direction']).reset_index(drop=True)

SIGNALS_DIR = Path('/kaggle/working/signals_v3')
SIGNALS_DIR.mkdir(parents=True, exist_ok=True)

cols = ['prediction_time','ticker','level_id','direction',
        'proba_1h','proba_4h','proba_24h','level_type_str',
        'center','zone_low','zone_high','price_position_in_zone',
        'touch_count','is_confirmation_touch']

for tk, sub in base.groupby('ticker'):
    sub.sort_values('prediction_time')[cols].to_csv(
        SIGNALS_DIR / f'signals_{tk}.csv', index=False)
    print(f'  {tk:12s} {len(sub):>6,} signals')

  ADAUSDT         393 signals
  BNBUSDT         330 signals
  ETHUSDT         333 signals


### SignalStrategy 

In [69]:
# ============================================================
# SignalStrategy v2 — dynamic TP + trailing stop
# ============================================================

class _VirtualLevel:
    __slots__ = ('kind','center','zone_low','zone_high',
                 'proba_1h','proba_4h','proba_24h','price_position_in_zone',
                 'used','confirmed_i','state','breakout_i',
                 'level_id','signal_time')

    def __init__(self, kind, center, zone_low, zone_high,
                 proba_1h, proba_4h, proba_24h, price_position_in_zone,
                 level_id, signal_time, confirmed_i):
        self.kind = kind
        self.center = float(center)
        self.zone_low = float(zone_low)
        self.zone_high = float(zone_high)
        self.proba_1h = float(proba_1h)
        self.proba_4h = float(proba_4h)
        self.proba_24h = float(proba_24h)
        self.price_position_in_zone = float(price_position_in_zone)
        self.level_id = level_id
        self.signal_time = signal_time
        self.used = False
        self.confirmed_i = confirmed_i
        self.state = 'waiting_breakout'
        self.breakout_i = None


class _SignalDetector:
    def __init__(self, signals_df):
        df = signals_df.copy()
        df['prediction_time'] = pd.to_datetime(df['prediction_time'], utc=True)
        self.signals = df.sort_values('prediction_time').reset_index(drop=True)
        self.cursor = 0
        self.levels = []

    def inject_due(self, current_time, current_bar):
        if current_time.tzinfo is None:
            current_time = current_time.tz_localize('UTC')
        n = len(self.signals)
        while self.cursor < n:
            row = self.signals.iloc[self.cursor]
            if row['prediction_time'] > current_time:
                break
            kind = 'resistance' if row['direction'] == 'long' else 'support'
            self.levels.append(_VirtualLevel(
                kind=kind,
                center=row['center'],
                zone_low=row['zone_low'],
                zone_high=row['zone_high'],
                proba_1h=row['proba_1h'],
                proba_4h=row['proba_4h'],
                proba_24h=row['proba_24h'],
                price_position_in_zone=row['price_position_in_zone'],
                level_id=row['level_id'],
                signal_time=row['prediction_time'],
                confirmed_i=current_bar,
            ))
            self.cursor += 1

    def get_active_levels(self):
        return self.levels


class SignalStrategy(Strategy):
    # ----- runtime -----
    signals_path = None

    # ----- entry -----
    entry_mode        = 'immediate'   # 'immediate' | 'breakout' | 'retest'
    retest_ttl_bars   = 20

    # ----- risk model -----
    stop_pct             = 0.03       # initial stop distance (fraction)
    min_r_multiple       = 3.0        # TP at least 3R
    use_natural_target   = True       # allow TP above 3R if a natural level exists
    target_lookback_bars = 240        # look back 240 bars (~4h) for natural target

    # ----- trailing -----
    move_to_breakeven_r = 1.0         # move stop to entry once +1R
    trail_activation_r  = 2.0         # start trailing once +2R
    trail_distance_r    = 0.5         # trail 0.5R behind the best price

    # ----- safety -----
    max_hold_hours    = 72            # hard time stop (fallback)

    # ----- filters -----
    use_ppz           = True
    long_ppz_min      = 0.7
    short_ppz_max     = 0.3
    trade_direction   = 'both'        # 'both' | 'long' | 'short'

    def init(self):
        sig = pd.read_csv(self.signals_path)
        self.detector      = _SignalDetector(sig)
        self.entry_price   = None
        self.stop_price    = None
        self.tp_price      = None
        self.risk_per_unit = None
        self.best_price    = None
        self._entry_bar    = None
        self._max_hold     = int(self.max_hold_hours * 60)

    # ----- filters -----
    def _direction_ok(self, lv):
        if self.trade_direction == 'both':
            return True
        if self.trade_direction == 'long':
            return lv.kind == 'resistance'
        return lv.kind == 'support'

    def _ppz_ok(self, lv):
        if not self.use_ppz:
            return True
        if lv.kind == 'resistance':
            return lv.price_position_in_zone >= self.long_ppz_min
        return lv.price_position_in_zone <= self.short_ppz_max

    # ----- helpers -----
    def _natural_target(self, direction, entry):
        """Highest high / lowest low over the lookback window (excl. current bar)."""
        if not self.use_natural_target:
            return None
        n = min(self.target_lookback_bars, len(self.data) - 1)
        if n < 10:
            return None
        window_high = float(np.max(self.data.High[-n:-1]))
        window_low  = float(np.min(self.data.Low[-n:-1]))
        if direction == 'long' and window_high > entry:
            return window_high
        if direction == 'short' and window_low < entry:
            return window_low
        return None

    def _enter(self, direction, close, bar):
        # 1) risk and initial stop
        risk = close * self.stop_pct
        if direction == 'long':
            stop = close - risk
        else:
            stop = close + risk

        # 2) minimum TP = min_r_multiple * risk
        if direction == 'long':
            tp = close + self.min_r_multiple * risk
        else:
            tp = close - self.min_r_multiple * risk

        # 3) extend TP if a natural target is further away
        nat = self._natural_target(direction, close)
        if nat is not None:
            if direction == 'long' and nat > tp:
                tp = nat
            elif direction == 'short' and nat < tp:
                tp = nat

        # 4) open position
        if direction == 'long':
            self.buy()
        else:
            self.sell()

        self.entry_price   = close
        self.stop_price    = stop
        self.tp_price      = tp
        self.risk_per_unit = risk
        self.best_price    = close
        self._entry_bar    = bar

    def _close(self):
        if self.position:
            self.position.close()
        self.entry_price   = None
        self.stop_price    = None
        self.tp_price      = None
        self.risk_per_unit = None
        self.best_price    = None
        self._entry_bar    = None

    # ----- main loop -----
    def next(self):
        close = float(self.data.Close[-1])
        high  = float(self.data.High[-1])
        low   = float(self.data.Low[-1])
        open_ = float(self.data.Open[-1])

        bar = len(self.data.Close) - 1
        now = self.data.index[-1]

        self.detector.inject_due(now, bar)
        levels = self.detector.get_active_levels()

        # --- direction gate ---
        if self.position:
            if self.trade_direction == 'long' and self.position.is_short:
                self._close()
            elif self.trade_direction == 'short' and self.position.is_long:
                self._close()

        # --- manage open position ---
        if self.position and self.entry_price is not None:
            direction = 'long' if self.position.is_long else 'short'

            # update best price in our favour
            if direction == 'long':
                self.best_price = max(self.best_price, high)
                profit_r = (self.best_price - self.entry_price) / self.risk_per_unit
            else:
                self.best_price = min(self.best_price, low)
                profit_r = (self.entry_price - self.best_price) / self.risk_per_unit

            # move stop to breakeven
            if profit_r >= self.move_to_breakeven_r:
                if direction == 'long':
                    self.stop_price = max(self.stop_price, self.entry_price)
                else:
                    self.stop_price = min(self.stop_price, self.entry_price)

            # activate trailing
            if profit_r >= self.trail_activation_r:
                if direction == 'long':
                    new_stop = self.best_price - self.trail_distance_r * self.risk_per_unit
                    self.stop_price = max(self.stop_price, new_stop)
                else:
                    new_stop = self.best_price + self.trail_distance_r * self.risk_per_unit
                    self.stop_price = min(self.stop_price, new_stop)

            # exit checks (order: TP first, then stop, then time)
            if direction == 'long':
                if high >= self.tp_price:
                    self._close(); return
                if low <= self.stop_price:
                    self._close(); return
            else:
                if low <= self.tp_price:
                    self._close(); return
                if high >= self.stop_price:
                    self._close(); return

            if self._entry_bar is not None and (bar - self._entry_bar) > self._max_hold:
                self._close(); return

        # --- look for entries ---
        for lv in levels:
            if lv.used or lv.confirmed_i is None:
                continue
            if not self._direction_ok(lv):
                lv.used = True; continue
            if not self._ppz_ok(lv):
                lv.used = True; continue

            level = lv.center
            direction = 'long' if lv.kind == 'resistance' else 'short'

            if self.entry_mode == 'immediate':
                if self.position:
                    continue
                self._enter(direction, close, bar)
                lv.used = True

            elif self.entry_mode == 'breakout':
                if self.position:
                    continue
                if lv.kind == 'resistance' and close > level:
                    self._enter('long', close, bar); lv.used = True
                elif lv.kind == 'support' and close < level:
                    self._enter('short', close, bar); lv.used = True

            elif self.entry_mode == 'retest':
                if lv.state == 'waiting_breakout':
                    if lv.kind == 'resistance' and close > level:
                        lv.state = 'waiting_retest'; lv.breakout_i = bar
                    elif lv.kind == 'support' and close < level:
                        lv.state = 'waiting_retest'; lv.breakout_i = bar
                elif lv.state == 'waiting_retest':
                    if bar - lv.breakout_i > self.retest_ttl_bars:
                        lv.state = 'waiting_breakout'
                        lv.breakout_i = None
                        continue
                    if not (low <= level <= high):
                        continue
                    if self.position:
                        continue
                    if lv.kind == 'resistance' and close > open_:
                        self._enter('long', close, bar); lv.used = True
                    elif lv.kind == 'support' and close < open_:
                        self._enter('short', close, bar); lv.used = True

###  Loading 1m klines is only available for focus tickers.

In [70]:
from tqdm.notebook import tqdm

ALL_COLUMNS = ['open_time','open','high','low','close','volume',
               'close_time','quote_volume','count','taker_base','taker_quote','ignore']
USE_COLS = ['open_time','open','high','low','close','volume','quote_volume']
col_idx  = [ALL_COLUMNS.index(c) for c in USE_COLS]


def load_klines_for_ticker(ticker, sig_path, base=KLINES_BASE):
    sig = pd.read_csv(sig_path)
    sig['prediction_time'] = pd.to_datetime(sig['prediction_time'], utc=True, errors='coerce')
    sig = sig.dropna(subset=['prediction_time'])
    if len(sig) == 0:
        return None
    t_min = sig['prediction_time'].min()
    t_max = sig['prediction_time'].max() + pd.Timedelta(hours=24)

    ticker_dir = os.path.join(base, ticker, '1m')
    if not os.path.isdir(ticker_dir):
        return None
    files = sorted(glob.glob(os.path.join(ticker_dir, '**', '*.csv'), recursive=True))

    months, cur = [], t_min.floor('D').replace(day=1)
    end = t_max.floor('D').replace(day=1)
    while cur <= end:
        months.append(cur.strftime('%Y-%m'))
        cur = cur.replace(year=cur.year + (cur.month == 12), month=(cur.month % 12) + 1)
    months = set(months)

    target = [f for f in files if any(m in os.path.basename(f) for m in months)] or files

    chunks = []
    for f in target:
        try:
            m = pd.read_csv(f, header=None, names=ALL_COLUMNS, usecols=col_idx)
            m['open_time'] = pd.to_numeric(m['open_time'], errors='coerce')
            m = m.dropna(subset=['open_time'])
            m['open_time'] = m['open_time'].apply(lambda x: x*1000 if x < 1e13 else x)
            m['open_time'] = pd.to_datetime(m['open_time'], unit='us', utc=True)
            chunks.append(m)
        except Exception:
            pass
    if not chunks:
        return None

    raw = pd.concat(chunks, ignore_index=True)
    num = [c for c in USE_COLS if c != 'open_time']
    raw[num] = raw[num].apply(pd.to_numeric, errors='coerce')
    raw = raw.sort_values('open_time').reset_index(drop=True)
    raw = raw.loc[(raw['open_time'] >= t_min) & (raw['open_time'] <= t_max)]
    raw = raw.drop_duplicates(subset='open_time', keep='first')
    return raw


def make_bt_df(raw):
    df = raw.rename(columns={
        'open_time':'Date','open':'Open','high':'High',
        'low':'Low','close':'Close','volume':'Volume',
    }).copy()
    df['Date'] = pd.to_datetime(df['Date'], utc=True)
    df = df.set_index('Date').sort_index()
    return df[~df.index.duplicated(keep='first')][['Open','High','Low','Close','Volume']]


klines_all = {}
for tk in tqdm(testable, desc='Loading klines'):
    sig_p = SIGNALS_DIR / f'signals_{tk}.csv'
    if not sig_p.exists():
        print(f'  {tk}: no signals'); continue
    raw = load_klines_for_ticker(tk, sig_p)
    if raw is None or len(raw) == 0:
        print(f'  {tk}: no klines'); continue
    klines_all[tk] = make_bt_df(raw)
    print(f'  {tk:12s} {len(raw):>10,} bars')

print(f'\nLoaded {len(klines_all)} / {len(testable)}')

Loading klines:   0%|          | 0/3 [00:00<?, ?it/s]

  ADAUSDT         557,275 bars
  BNBUSDT         553,393 bars
  ETHUSDT         552,513 bars

Loaded 3 / 3


In [71]:
combos = list(itertools.product(STRATEGY_GRID, PROBA_GRID))
print(f'Combos : {len(combos)}   Tickers: {len(klines_all)}   '
      f'Total runs: {len(combos) * len(klines_all)}\n')

rows = []
for strat, proba_thr in combos:
    for tk in klines_all:
        sig = pd.read_csv(SIGNALS_DIR / f'signals_{tk}.csv')
        sig['prediction_time'] = pd.to_datetime(sig['prediction_time'], utc=True)
        sub = sig[sig[PROBA_COL] >= proba_thr].reset_index(drop=True)
        if len(sub) < 5:
            continue

        tmp = SIGNALS_DIR / f'_tmp_{tk}.csv'
        sub.to_csv(tmp, index=False)

        bt = Backtest(klines_all[tk], SignalStrategy,
                      cash=CASH, commission=COMMISSION, exclusive_orders=True)
        try:
            st = bt.run(
                signals_path          = str(tmp),
                entry_mode            = strat['entry_mode'],
                stop_pct              = strat['stop_pct'],
                min_r_multiple        = strat['min_r_multiple'],
                move_to_breakeven_r   = strat['move_to_breakeven_r'],
                trail_activation_r    = strat['trail_activation_r'],
                trail_distance_r      = strat['trail_distance_r'],
                use_ppz               = USE_PPZ,
                trade_direction       = TRADE_DIR,
            )
        except Exception as e:
            tmp.unlink()
            print(f'  error {tk}: {e}')
            continue
        tmp.unlink()

        n = int(st['# Trades'])
        if n < 3:
            continue

        rows.append({
            'ticker':              tk,
            'entry_mode':          strat['entry_mode'],
            'stop_pct':            strat['stop_pct'],
            'min_r_multiple':      strat['min_r_multiple'],
            'move_to_breakeven_r': strat['move_to_breakeven_r'],
            'trail_activation_r':  strat['trail_activation_r'],
            'trail_distance_r':    strat['trail_distance_r'],
            'threshold':           proba_thr,
            'n_signals':           len(sub),
            'trades':              n,
            'return_%':            round(float(st['Return [%]']), 2),
            'pf':                  round(float(st['Profit Factor']), 2),
            'win_rate_%':          round(float(st['Win Rate [%]']), 2),
            'max_dd_%':            round(float(st['Max. Drawdown [%]']), 2),
            'sharpe':              round(float(st['Sharpe Ratio']) if n > 1 else 0, 2),
        })

    print(f'  done: {strat["entry_mode"]:10s} stop={strat["stop_pct"]:.2f} '
          f'R:R={strat["min_r_multiple"]:.1f} thr={proba_thr:.2f}  rows={len(rows)}')

results = pd.DataFrame(rows)
results.to_csv('/kaggle/working/backtest_grid_v2.csv', index=False)
print(f'\nSaved {len(results)} rows')
display(results.sort_values('pf', ascending=False).head(20))

Combos : 9   Tickers: 3   Total runs: 27



Backtest.run:   0%|          | 0/557274 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/553392 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/552512 [00:00<?, ?bar/s]

  done: immediate  stop=0.03 R:R=3.0 thr=0.24  rows=3


...

Backtest.run:   0%|          | 0/557274 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/553392 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/552512 [00:00<?, ?bar/s]

  done: breakout   stop=0.05 R:R=4.0 thr=0.45  rows=18
  done: breakout   stop=0.05 R:R=4.0 thr=0.65  rows=18

Saved 18 rows


,ticker,entry_mode,stop_pct,min_r_multiple,move_to_breakeven_r,trail_activation_r,trail_distance_r,threshold,n_signals,trades,return_%,pf,win_rate_%,max_dd_%,sharpe
15,ADAUSDT,breakout,0.05,4.0,1.0,2.0,0.5,0.45,142,61,-9.89,1.01,36.07,-41.85,-0.24
9,ADAUSDT,immediate,0.05,3.0,1.0,2.0,0.5,0.45,142,60,-21.56,0.92,33.33,-43.53,-0.57
17,ETHUSDT,breakout,0.05,4.0,1.0,2.0,0.5,0.45,115,52,-19.62,0.89,36.54,-37.65,-0.66
6,ADAUSDT,immediate,0.05,3.0,1.0,2.0,0.5,0.24,393,115,-59.82,0.78,34.78,-71.23,-2.19
4,BNBUSDT,immediate,0.03,3.0,1.0,2.0,0.5,0.45,122,63,-23.96,0.77,30.16,-34.91,-1.15
7,BNBUSDT,immediate,0.05,3.0,1.0,2.0,0.5,0.24,330,97,-41.62,0.75,41.24,-47.60,-1.54
12,ADAUSDT,breakout,0.05,4.0,1.0,2.0,0.5,0.24,393,120,-71.12,0.73,31.67,-76.39,-3.31
11,ETHUSDT,immediate,0.05,3.0,1.0,2.0,0.5,0.45,115,51,-29.44,0.68,31.37,-47.18,-1.31
1,BNBUSDT,immediate,0.03,3.0,1.0,2.0,0.5,0.24,330,115,-57.84,0.58,28.70,-60.52,-3.77
14,ETHUSDT,breakout,0.05,4.0,1.0,2.0,0.5,0.24,333,95,-65.62,0.57,30.53,-70.96,-3.73


In [ ]:
# Let’s take the best one in terms of PF.
best = results.sort_values('pf', ascending=False).iloc[0]
print('Best combo:')
print(best[['ticker','entry_mode','stop_pct','min_r_multiple',
            'move_to_breakeven_r','trail_activation_r',
            'trail_distance_r','threshold','pf','return_%','win_rate_%']].to_dict())

tk = best['ticker']
sig = pd.read_csv(SIGNALS_DIR / f'signals_{tk}.csv')
sig['prediction_time'] = pd.to_datetime(sig['prediction_time'], utc=True)
sub = sig[sig[PROBA_COL] >= best['threshold']].reset_index(drop=True)

tmp = SIGNALS_DIR / f'_tmp_{tk}_plot.csv'
sub.to_csv(tmp, index=False)

bt = Backtest(klines_all[tk], SignalStrategy,
              cash=CASH, commission=COMMISSION, exclusive_orders=True)

stats = bt.run(
    signals_path          = str(tmp),
    entry_mode            = best['entry_mode'],
    stop_pct              = best['stop_pct'],
    min_r_multiple        = best['min_r_multiple'],
    move_to_breakeven_r   = best['move_to_breakeven_r'],
    trail_activation_r    = best['trail_activation_r'],
    trail_distance_r      = best['trail_distance_r'],
    use_ppz               = USE_PPZ,
    trade_direction       = TRADE_DIR,
)
tmp.unlink()

# --- chart ---
bt.plot(
    filename='/kaggle/working/best_backtest_v2.html',
    open_browser=False,
)

# --- table trades ---
trades = stats['_trades']
print(f'\nTotal trades : {len(trades)}')
if len(trades):
    print(f'Mean PnL     : {trades["PnL"].mean():.2f}')
    print(f'Mean return  : {trades["ReturnPct"].mean():.2%}')
    print(f'Win rate     : {(trades["PnL"] > 0).mean():.2%}')
    print(f'Mean duration: {trades["Duration"].mean()}')
    display(trades[['EntryTime','ExitTime','Size','EntryPrice','ExitPrice',
                    'PnL','ReturnPct','Duration']].head(20))

Best combo:
{'ticker': 'ADAUSDT', 'entry_mode': 'breakout', 'stop_pct': 0.05, 'min_r_multiple': 4.0, 'move_to_breakeven_r': 1.0, 'trail_activation_r': 2.0, 'trail_distance_r': 0.5, 'threshold': 0.45, 'pf': 1.01, 'return_%': -9.89, 'win_rate_%': 36.07}


Backtest.run:   0%|          | 0/557274 [00:00<?, ?bar/s]


Total trades : 61
Mean PnL     : -16.21
Mean return  : 0.03%
Win rate     : 36.07%
Mean duration: 1 days 13:23:17.704918032


,EntryTime,ExitTime,Size,EntryPrice,ExitPrice,PnL,ReturnPct,Duration
0,2025-01-03 16:22:00+00:00,2025-01-04 10:11:00+00:00,8981,1.1129,1.0566,-515.372440,-0.051563,0 days 17:49:00
1,2025-01-07 08:30:00+00:00,2025-01-07 15:16:00+00:00,8419,1.1259,1.0604,-560.647730,-0.059147,0 days 06:46:00
2,2025-01-19 08:28:00+00:00,2025-01-20 06:50:00+00:00,-8490,1.0505,1.0497,-2.123349,-0.000238,0 days 22:22:00
3,2025-01-22 18:58:00+00:00,2025-01-25 18:59:00+00:00,-9102,0.9797,0.9835,-43.522123,-0.004881,3 days 00:01:00
4,2025-01-26 05:50:00+00:00,2025-01-27 00:04:00+00:00,8972,0.9890,0.9456,-398.063416,-0.044861,0 days 18:14:00
5,2025-01-31 21:31:00+00:00,2025-02-02 18:26:00+00:00,-9061,0.9354,0.8207,1031.340689,0.121683,1 days 20:55:00
6,2025-02-26 17:58:00+00:00,2025-02-28 07:19:00+00:00,-14534,0.6541,0.5976,812.074896,0.085421,1 days 13:21:00
7,2025-03-09 02:59:00+00:00,2025-03-09 19:32:00+00:00,-13000,0.7937,0.7304,812.993350,0.078793,0 days 16:33:00
8,2025-03-10 01:05:00+00:00,2025-03-10 10:22:00+00:00,15015,0.7413,0.7415,-8.129121,-0.000730,0 days 09:17:00
9,2025-03-19 13:16:00+00:00,2025-03-22 13:17:00+00:00,15216,0.7310,0.7084,-354.832555,-0.031901,3 days 00:01:00


In [73]:
bt.plot()

GridPlot(id='p2766', ...)

In [74]:
stats

Start                     2024-12-09 21:05...
End                       2025-12-31 20:59...
Duration                    386 days 23:54:00
Exposure Time [%]                    24.56633
Equity Final [$]                   9011.08847
Equity Peak [$]                   12357.00758
Commissions [$]                     574.47483
Return [%]                           -9.88912
Buy & Hold Return [%]               -63.87474
Return (Ann.) [%]                    -9.35412
Volatility (Ann.) [%]                38.93842
CAGR [%]                             -9.36032
Sharpe Ratio                         -0.24023
Sortino Ratio                        -0.35966
Calmar Ratio                         -0.22353
Alpha [%]                           -12.51247
Beta                                 -0.04107
Max. Drawdown [%]                   -41.84719
Avg. Drawdown [%]                    -1.87066
Max. Drawdown Duration      233 days 09:15:00
Avg. Drawdown Duration        4 days 14:02:00
# Trades                          